In [1]:
pip install delta-spark==2.4.0

24.58s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, when, trim, to_date, year
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Caminho base do warehouse (DO SEU AMBIENTE)
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

# Criar SparkSession com Hive (COMO NO SEU SCRIPT)
spark = (
    SparkSession.builder
    .appName("RottenTomatoes ETL")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

# --- 1. EXTRAÇÃO (Extract) ---

# Definir o schema manualmente
titles_schema = StructType([
    StructField("movie_title", StringType(), True),
    StructField("movie_info", StringType(), True),
    StructField("critics_consensus", StringType(), True),
    StructField("rating", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("directors", StringType(), True),
    StructField("writers", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("in_theaters_date", StringType(), True), # Lemos como string
    StructField("on_streaming_date", StringType(), True), # Lemos como string
    StructField("runtime_in_minutes", IntegerType(), True),
    StructField("studio_name", StringType(), True),
    StructField("tomatometer_status", StringType(), True),
    StructField("tomatometer_rating", DoubleType(), True),
    StructField("tomatometer_count", DoubleType(), True),
    StructField("audience_rating", DoubleType(), True),
    StructField("audience_count", DoubleType(), True)
])

# Caminho do ficheiro bronze (AJUSTE CONFORME NECESSÁRIO)
csv_path = "hdfs://hdfs-nn:9000/demo/bronze/Rotten Tomatoes Movies.csv"

# Ler o CSV de forma robusta
df = spark.read.csv(
    csv_path,
    header=True,
    schema=titles_schema,
    multiLine=True,
    escape='"',
    quote='"',
    nullValue="",
    emptyValue=None
)

print(f"Total de registos lidos: {df.count()}")
df.show(5, truncate=50)

# --- 2. TRANSFORMAÇÃO (Transform) ---

print("\n=== Início da Limpeza e Transformação ===")

# ==========================================================
# 2.1. TRATAMENTO CRÍTICO (Filtro 'in_theaters_date')
# ==========================================================
df_clean = df.withColumn("in_theaters_date_str", trim(col("in_theaters_date")))
total_antes_filtro = df_clean.count()
df_clean = df_clean.filter(
    col("in_theaters_date_str").isNotNull() & (col("in_theaters_date_str") != "")
)
total_depois_filtro = df_clean.count()
print(f"TRATAMENTO (Filtro 'in_theaters_date'): Removidos {total_antes_filtro - total_depois_filtro} registos por data de estreia nula ou vazia.")


# ==========================================================
# 2.2. TRATAMENTO (Conversão de Datas e Chave de Partição)
# ==========================================================
# Converter a string da data para o tipo Date (formato YYYY-MM-DD)
df_clean = df_clean.withColumn("in_theaters_date", to_date(col("in_theaters_date_str"), "yyyy-MM-dd"))

# <<< MUDANÇA AQUI >>>
# Converter 'on_streaming_date' para o tipo Date.
# O 'trim' remove espaços extra que possam impedir a conversão.
df_clean = df_clean.withColumn("on_streaming_date", to_date(trim(col("on_streaming_date")), "yyyy-MM-dd"))

# Criar a coluna 'release_year' (ano) a partir da data. Esta será a chave de partição.
df_clean = df_clean.withColumn("release_year", year(col("in_theaters_date")))

# Segundo filtro crítico: remover linhas onde a conversão da data de estreia falhou
total_antes_filtro_ano = df_clean.count()
df_clean = df_clean.filter(col("release_year").isNotNull())
total_depois_filtro_ano = df_clean.count()
if (total_antes_filtro_ano - total_depois_filtro_ano) > 0:
    print(f"TRATAMENTO (Filtro 'release_year'): Removidos {total_antes_filtro_ano - total_depois_filtro_ano} registos por formato de data inválido.")

# ==========================================================
# 2.3. TRATAMENTO (Limpeza Geral de Nulos)
# ==========================================================
# Tratar strings que são apenas espaços como nulos
string_cols = [c.name for c in df_clean.schema.fields if c.dataType == StringType()]
for column in string_cols:
    df_clean = df_clean.withColumn(column, when(trim(col(column)) == "", None).otherwise(col(column)))
print("TRATAMENTO: Convertidas strings vazias (só espaços) para nulo.")

# <<< MUDANÇA AQUI >>>
# Substituir valores nulos por defaults.
# 'on_streaming_date' foi REMOVIDA daqui. Ela ficará NULL (DateType) se a data não for válida.
df_clean = df_clean.fillna({
    # Strings
    "movie_info": "Sem informação",
    "critics_consensus": "Sem informação",
    "rating": "Não classificado",
    "genre": "Desconhecido",
    "directors": "Desconhecido",
    "writers": "Desconhecido",
    "cast": "Desconhecido",
    "studio_name": "Desconhecido",
    "tomatometer_status": "Desconhecido",
    
    # Numéricos
    "runtime_in_minutes": 0,
    "tomatometer_rating": 0.0,
    "tomatometer_count": 0.0,
    "audience_rating": 0.0,
    "audience_count": 0.0
})
print("TRATAMENTO: Preenchidos valores nulos (strings e numéricos) com padrões.")


# --- 3. CARGA (Load) ---

# Caminho para camada Silver
silver_path = f"{warehouse_location}/silver.db/rotten_tomatoes"

# Criar base de dados Silver se nao existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

print(f"\n=== A guardar dados limpos em {silver_path} ===")

# Guardar em formato Parquet particionado por ano
df_clean.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("release_year") \
    .save(silver_path)

# Criar tabela Hive externa associada ao Parquet
spark.sql("DROP TABLE IF EXISTS silver.rotten_tomatoes")

# <<< MUDANÇA AQUI >>>
# Atualizado o tipo da coluna 'on_streaming_date' para DATE
spark.sql(f"""
    CREATE EXTERNAL TABLE silver.rotten_tomatoes (
        movie_title STRING,
        movie_info STRING,
        critics_consensus STRING,
        rating STRING,
        genre STRING,
        directors STRING,
        writers STRING,
        cast STRING,
        in_theaters_date DATE,
        on_streaming_date DATE, 
        runtime_in_minutes INT,
        studio_name STRING,
        tomatometer_status STRING,
        tomatometer_rating DOUBLE,
        tomatometer_count DOUBLE,
        audience_rating DOUBLE,
        audience_count DOUBLE
    )
    PARTITIONED BY (release_year INT)
    STORED AS PARQUET
    LOCATION '{silver_path}'
""")

# Atualizar partições (como no seu script original)
spark.catalog.recoverPartitions("silver.rotten_tomatoes")

print("\n Tabela Parquet particionada criada com sucesso em silver.rotten_tomatoes")

# Exemplo de query
print("\n=== Contagem por ano ===")
spark.sql("""
    SELECT release_year, COUNT(*) AS total_titulos
    FROM silver.rotten_tomatoes
    GROUP BY release_year
    ORDER BY release_year DESC
    LIMIT 20
""").show()

print("\n=== Verificação de Datas de Streaming (Null vs Válidas) ===")
spark.sql("""
    SELECT 
        COUNT(*) AS total,
        COUNT(CASE WHEN on_streaming_date IS NULL THEN 1 END) AS com_streaming_nulo,
        COUNT(CASE WHEN on_streaming_date IS NOT NULL THEN 1 END) AS com_streaming_valido
    FROM silver.rotten_tomatoes
""").show()


spark.stop()

Total de registos lidos: 16638
+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------+--------------------------------------------------+-----------------+-----------------+--------------------------------------------------+----------------+-----------------+------------------+----------------------+------------------+------------------+-----------------+---------------+--------------+
|                                       movie_title|                                        movie_info|                                 critics_consensus|rating|                                             genre|        directors|          writers|                                              cast|in_theaters_date|on_streaming_date|runtime_in_minutes|           studio_name|tomatometer_status|tomatometer_rating|tomatometer_count|audience_rating|audience_count|
+----------------------------------------

In [5]:
from pyspark.sql import SparkSession

# --- 1. CONFIGURAÇÃO DA SPARKSESSION (com Delta) ---
# Esta célula precisa de ter a sua própria SparkSession que 
# carrega as bibliotecas do Delta.
print("A iniciar a SparkSession com o pacote Delta Lake...")

# Define as variáveis do teu ambiente
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
hive_uris = "thrift://hive-metastore:9083"

spark = (
    SparkSession.builder
    .appName("Delta Converter") # Um novo nome para a app
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", hive_uris)
    
    # --- Configuração essencial do Delta ---
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    
    .enableHiveSupport()
    .getOrCreate()
)

print("Sessão Spark com Delta pronta.")


# --- 2. O COMANDO DE CONVERSÃO ---

print("A converter a tabela Parquet 'silver.rotten_tomatoes' para Delta Lake...")

# Este é o comando "mágico". Ele não re-escreve os teus dados Parquet.
# Ele apenas adiciona o '_delta_log' para "adotar" os ficheiros existentes.
# Como a tua tabela original está particionada, és obrigado a dizê-lo ao Delta.
try:
    spark.sql("""
        CONVERT TO DELTA silver.rotten_tomatoes
        PARTITIONED BY (release_year INT)
    """)
    print(" Conversão concluída com sucesso!")

except Exception as e:
    print(f"❌ Ocorreu um erro durante a conversão. Verifica se a tabela 'silver.rotten_tomatoes' existe e é Parquet.")
    print(e)


# (Opcional) Podes parar esta sessão se não fores fazer mais nada
# spark.stop()

A iniciar a SparkSession com o pacote Delta Lake...
Sessão Spark com Delta pronta.
A converter a tabela Parquet 'silver.rotten_tomatoes' para Delta Lake...
✅ Conversão concluída com sucesso!
